# 09 Reset Data

## Purpose

Delete all generated data files under `data/` so that the full notebook pipeline (01–08) can be re-run from a clean state. All directories are preserved; only files are removed.

## Inputs

All files currently present under `data/bronze/`, `data/silver/`, `data/gold/`, `data/samples/`, and `data/checkpoints/`.

## Outputs

Empty directory tree under `data/`. No Parquet, CSV, JSON, HTML, or JSONL files remain.
Re-running notebooks 01–08 in order will repopulate all directories.

## Configuration

Set `DRY_RUN=true` (env var or cell variable below) to preview which files would be deleted without actually removing them.

In [1]:
from pathlib import Path
import os

PROJECT_ROOT = (Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd())
DATA_DIR = PROJECT_ROOT / Path(os.getenv("DATA_DIR", "data"))

DRY_RUN = os.getenv("DRY_RUN", "false").lower() == "true"

MANAGED_DIRS = [
    DATA_DIR / "bronze" / "eea",
    DATA_DIR / "bronze" / "open_meteo_raw",
    DATA_DIR / "bronze" / "wikipedia_html",
    DATA_DIR / "bronze",
    DATA_DIR / "silver",
    DATA_DIR / "gold",
    DATA_DIR / "samples",
    DATA_DIR / "checkpoints",
    DATA_DIR,
]

print(f"PROJECT_ROOT : {PROJECT_ROOT}")
print(f"DATA_DIR     : {DATA_DIR}")
print(f"DRY_RUN      : {DRY_RUN}")

PROJECT_ROOT : C:\dev\euro-air-quality-pipeline
DATA_DIR     : C:\dev\euro-air-quality-pipeline\data
DRY_RUN      : False


## Implementation

### Step 1 — Preview files to be deleted

In [2]:
import pandas as pd

all_files = sorted(DATA_DIR.rglob("*"))
file_records = [
    {
        "path": str(p.relative_to(PROJECT_ROOT)),
        "size_bytes": p.stat().st_size,
        "type": p.suffix or "(no ext)",
    }
    for p in all_files
    if p.is_file()
]

preview_df = pd.DataFrame(file_records) if file_records else pd.DataFrame(columns=["path", "size_bytes", "type"])
print(f"{len(preview_df)} Datei(en) werden {'VORSCHAU (DRY_RUN)' if DRY_RUN else 'gelöscht'}:")
preview_df

23 Datei(en) werden gelöscht:


,path,size_bytes,type
0,data\bronze\eea\.gitkeep,0,(no ext)
1,data\bronze\open_meteo_raw\.gitkeep,0,(no ext)
2,data\bronze\open_meteo_raw\amsterdam_nl.json,2050,.json
3,data\bronze\open_meteo_raw\berlin_de.json,1998,.json
4,data\bronze\open_meteo_raw\madrid_es.json,1735,.json
5,data\bronze\open_meteo_raw\mock_kafka_air_qual...,63279,.jsonl
6,data\bronze\open_meteo_raw\open_meteo_air_qual...,63279,.jsonl
7,data\bronze\open_meteo_raw\open_meteo_ingestio...,3513,.json
8,data\bronze\open_meteo_raw\paris_fr.json,2006,.json
9,data\bronze\open_meteo_raw\prague_cz.json,2013,.json


### Step 2 — Delete all files, keep directory structure

In [3]:
deleted = []
skipped = []

for p in sorted(DATA_DIR.rglob("*"), reverse=True):
    if not p.is_file():
        continue
    if DRY_RUN:
        skipped.append(str(p.relative_to(PROJECT_ROOT)))
    else:
        p.unlink()
        deleted.append(str(p.relative_to(PROJECT_ROOT)))

if DRY_RUN:
    print(f"DRY RUN — {len(skipped)} Datei(en) würden gelöscht, keine Änderung.")
else:
    print(f"{len(deleted)} Datei(en) gelöscht.")

# Ensure all expected directories still exist
for d in MANAGED_DIRS:
    d.mkdir(parents=True, exist_ok=True)

23 Datei(en) gelöscht.


## Validation / Quality Checks

Confirm that no files remain under `data/` and that all expected directories are present.

In [4]:
remaining_files = [p for p in DATA_DIR.rglob("*") if p.is_file()]

if not DRY_RUN:
    assert remaining_files == [], f"Noch {len(remaining_files)} Datei(en) vorhanden: {remaining_files[:5]}"

for d in MANAGED_DIRS:
    assert d.exists() and d.is_dir(), f"Verzeichnis fehlt nach Reset: {d}"

status = "DRY RUN — keine Änderung" if DRY_RUN else f"Reset abgeschlossen. {len(deleted)} Datei(en) gelöscht."
print(status)

dir_summary = pd.DataFrame(
    [{"directory": str(d.relative_to(PROJECT_ROOT)), "exists": d.exists()} for d in MANAGED_DIRS]
)
dir_summary

Reset abgeschlossen. 23 Datei(en) gelöscht.


,directory,exists
0,data\bronze\eea,True
1,data\bronze\open_meteo_raw,True
2,data\bronze\wikipedia_html,True
3,data\bronze,True
4,data\silver,True
5,data\gold,True
6,data\samples,True
7,data\checkpoints,True
8,data,True


## Results

After execution the `data/` tree is empty. Re-run notebooks 01–08 in order to rebuild all Bronze, Silver, Gold and sample outputs.

## Limitations

Real EEA source files placed manually under `data/bronze/eea/` are also deleted. If you have real EEA data, copy it back before running notebook 03.